# first

Every earlier notebook (`First_agent`, `Multiple_inputs`, `Sequential_Agent`) followed one fixed path through the graph. This notebook introduces **branching**: the graph picks *which* node runs next based on the state, using `add_conditional_edges` instead of a plain `add_edge`.

In [ ]:
from typing import Dict, TypedDict 
from langgraph.graph import StateGraph, START, END

New imports: `START` and `END` are sentinel node names LangGraph provides — `START` marks the graph's entry, `END` marks a path's exit. Earlier notebooks used `set_entry_point`/`set_finish_point` (or relied on an implicit finish); here they're used explicitly via `add_edge(START, ...)` and `add_edge(..., END)`, which is the more common style once a graph has branches.

In [ ]:
class AgentState(TypedDict):
    number1: int
    operation: str
    number2: int
    finalNumber: int


def adder(state: AgentState) -> AgentState:
    """This node add 2 numbers"""
    state["finalNumber"] = state["number1"] + state["number2"]
    return state

def subtractor(state: AgentState) -> AgentState:
    """This node subtracts the 2 numbers"""
    state["finalNumber"] = state["number1"] - state["number2"]
    return state

def decide_next_node(state: AgentState) -> AgentState:
    """This node will select the next node of the graph"""
    if state["operation"] == "+":
        return "addition_operation"
    elif state["operation"] == "-":
        return "subtraction_operation"




**`decide_next_node` is a router function, not a regular node.** Unlike `adder`/`subtractor`, it doesn't return an updated `state` — it returns a plain string ("addition_operation" / "subtraction_operation") that LangGraph uses to pick the next node. Its `-> AgentState` type hint is misleading here; the real return type is `str`.

Also note there's no `else` branch: if `state["operation"]` is anything other than `"+"` or `"-"`, this function implicitly returns `None`, which — as we saw when debugging the "Second" section earlier — makes LangGraph raise `KeyError: None` rather than silently doing nothing.

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("add_node",adder)
graph.add_node("subtract_node",subtractor)
graph.add_node("router", lambda state:state)

graph.add_edge(START,"router")
graph.add_conditional_edges(
    "router",
    decide_next_node,
    {
        "addition_operation": "add_node",
        "subtraction_operation": "subtract_node"
    }
)

graph.add_edge("add_node", END)
graph.add_edge("subtract_node", END)

app = graph.compile()

**`add_conditional_edges(source, router_fn, path_map)`:** after `source` runs, LangGraph calls `router_fn(state)` and looks up its return value in `path_map` to find the next node name. `router` itself is a no-op passthrough node (`lambda state: state`) — it exists purely so there's a named node to hang the conditional edges off of; the branching decision happens in `decide_next_node`, not inside `router`.

Here the finish is explicit: `add_edge("add_node", END)` and `add_edge("subtract_node", END)`, one per branch — contrast with `Sequential_Agent.ipynb`, where the single terminal node's finish was implicit.

In [ ]:
from IPython.display import Image,display
display(Image(app.get_graph().draw_mermaid_png()))

The diagram should now show a fork after `router`: two possible paths (`add_node` or `subtract_node`) both leading to `END` — the visual signature of a conditional edge, versus the single straight line from earlier notebooks.

In [ ]:
result = app.invoke({"number1": 10, "number2": 7, "operation": "+"})
result["finalNumber"]

Trace: `operation` is `"+"` → `decide_next_node` returns `"addition_operation"` → the path map sends execution to `add_node` → `adder` computes `10 + 7 = 17`. Try changing `operation` to `"-"` and re-running to see the other branch fire instead.

# Second

Same branching idea, chained twice: a first fork (`router` → add/subtract) feeds into a second fork (`router2` → add2/subtract2). Structurally this is `router → {add_node|subtract_node} → router2 → {add_node2|subtract_node2} → END` — two independent decisions in sequence, not a loop back to the start, so nothing here can run forever.

In [ ]:
class AgentState(TypedDict):
    number1: int
    operation: str
    number2: int
    finalNumber: int
    number3: int
    operation2: str
    number4: int
    finalNumber2: int


def adder(state: AgentState) -> AgentState:
    """This node add 2 numbers"""
    state["finalNumber"] = state["number1"] + state["number2"]
    return state

def subtractor(state: AgentState) -> AgentState:
    """This node subtracts the 2 numbers"""
    state["finalNumber"] = state["number1"] - state["number2"]
    return state

def decide_next_node(state: AgentState) -> AgentState:
    """This node will select the next node of the graph"""
    if state["operation"] == "+":
        return "addition_operation"
    elif state["operation"] == "-":
        return "subtraction_operation"


def adder2(state: AgentState) -> AgentState:
    """This node add 2 numbers"""
    state["finalNumber2"] = state["number3"] + state["number4"]
    return state

def subtractor2(state: AgentState) -> AgentState:
    """This node subtracts the 2 numbers"""
    state["finalNumber2"] = state["number3"] - state["number4"]
    return state

def decide_next_node2(state: AgentState) -> AgentState:
    """This node will select the next node of the graph"""
    if state["operation2"] == "+":
        return "addition_operation2"
    elif state["operation2"] == "-":
        return "subtraction_operation2"




`adder`, `subtractor`, and `decide_next_node` are **redefined** here with identical bodies to the "first" section. Because a notebook runs everything in one shared kernel namespace, this reassigns the same global names — the first section's graph (already compiled into its own `app`) keeps working since it captured references at build time, but if you re-run the first section's cells *after* this one, they'd end up using these (identical, so harmless here) redefinitions instead. New fields `number3`/`operation2`/`number4`/`finalNumber2` and the `adder2`/`subtractor2`/`decide_next_node2` trio drive the second decision point.

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("add_node",adder)
graph.add_node("subtract_node",subtractor)
graph.add_node("router", lambda state:state)
graph.add_node("add_node2",adder2)
graph.add_node("subtract_node2",subtractor2)
graph.add_node("router2", lambda state:state)

graph.add_edge(START,"router")


graph.add_conditional_edges(
    "router",
    decide_next_node,
    {
        "addition_operation": "add_node",
        "subtraction_operation": "subtract_node"
    }
)

graph.add_edge("add_node","router2")
graph.add_edge("subtract_node","router2")

graph.add_conditional_edges(
    "router2",
    decide_next_node2,
    {
        "addition_operation2": "add_node2",
        "subtraction_operation2": "subtract_node2"
    }
)

# graph.add_edge("add_node", "router2")
# graph.add_edge("subtract_node", "router2")
graph.add_edge("add_node2", END)
graph.add_edge("subtract_node2", END)

app = graph.compile()

Both branches of the first fork converge before the second one starts: `add_edge("add_node", "router2")` and `add_edge("subtract_node", "router2")` both point at the same next node, so whichever branch ran, execution continues into the second router. The commented-out `# graph.add_edge("add_node", "router2")` / `# graph.add_edge("subtract_node", "router2")` lines just below are leftover duplicates of the active ones above — dead code, safe to delete.

In [ ]:
from IPython.display import Image,display
display(Image(app.get_graph().draw_mermaid_png()))

The diagram now shows two diamonds in sequence — the first fork's two branches merging back into `router2` before it forks again. That merge-then-fork shape is the visual tell for "two chained decisions" versus one.

In [ ]:
initial_state = AgentState(number1 = 10, operation="-", number2 = 5, number3 = 7, number4=2, operation2="+", finalNumber= 0, finalNumber2 = 0)

Earlier notebooks passed a plain `dict` literal to `invoke`. Here `AgentState(...)` is called like a constructor with keyword args — this works because a `TypedDict` *is* just a `dict` at runtime (the typing is purely static/editor-level), so `AgentState(number1=10, ...)` produces an ordinary dict with those keys. Both styles are equivalent; this one gets you keyword-argument autocomplete for the state's fields.

In [ ]:
print(app.invoke(initial_state))

Trace: `operation="-"` routes to `subtract_node` → `finalNumber = 10 - 5 = 5`; then `operation2="+"` routes to `add_node2` → `finalNumber2 = 7 + 2 = 9`. Two independent branch decisions, each resolved once, then straight to `END` — matching the result dict above.